In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder


### Load Dataset

In [2]:
df = pd.read_csv("../data/processed/f1_eda_ready.csv") 
df.head()

,race_id,year,round,race_date,race_name,circuit_id,driver_id,driver_code,driver_name,constructor_id,constructor_name,grid_position,finish_position,points,status_id,quali_position,winner,dnf_flag
0,989,2018,1,2018-03-25,Australian Grand Prix,albert_park,20,vettel,Vettel,6,Ferrari,3,1,25.0,1,3.0,1,0
1,989,2018,1,2018-03-25,Australian Grand Prix,albert_park,1,hamilton,Hamilton,131,Mercedes,1,2,18.0,1,1.0,0,0
2,989,2018,1,2018-03-25,Australian Grand Prix,albert_park,8,raikkonen,Räikkönen,6,Ferrari,2,3,15.0,1,2.0,0,0
3,989,2018,1,2018-03-25,Australian Grand Prix,albert_park,817,ricciardo,Ricciardo,9,Red Bull,8,4,12.0,1,5.0,0,0
4,989,2018,1,2018-03-25,Australian Grand Prix,albert_park,4,alonso,Alonso,1,McLaren,10,5,10.0,1,11.0,0,0


### Sort Data Properly

VERY IMPORTANT.

Rolling features require chronological ordering.

In [3]:
df = df.sort_values(by=["driver_name", "year", "round"]).reset_index(drop=True)
df.head()

,race_id,year,round,race_date,race_name,circuit_id,driver_id,driver_code,driver_name,constructor_id,constructor_name,grid_position,finish_position,points,status_id,quali_position,winner,dnf_flag
0,1046,2020,16,2020-12-06,Sakhir Grand Prix,bahrain,851,aitken,Aitken,3,Williams,17,16,0.0,1,18.0,0,1
1,1010,2019,1,2019-03-17,Australian Grand Prix,albert_park,848,albon,Albon,5,Toro Rosso,13,14,0.0,11,13.0,0,0
2,1011,2019,2,2019-03-31,Bahrain Grand Prix,bahrain,848,albon,Albon,5,Toro Rosso,12,9,2.0,1,12.0,0,0
3,1013,2019,4,2019-04-28,Azerbaijan Grand Prix,baku,848,albon,Albon,5,Toro Rosso,11,11,0.0,11,12.0,0,0
4,1014,2019,5,2019-05-12,Spanish Grand Prix,catalunya,848,albon,Albon,5,Toro Rosso,11,11,0.0,1,12.0,0,0


### Driver Recent Form

Average finish position over last 3 races.

Lower = better.

In [4]:
df['driver_avg_finish_last3'] = df.groupby('driver_name')["finish_position"].transform(
                                lambda x : round(x.shift(1).rolling(3, min_periods=1).mean(),2)
                               )
df.head()

,race_id,year,round,race_date,race_name,circuit_id,driver_id,driver_code,driver_name,constructor_id,constructor_name,grid_position,finish_position,points,status_id,quali_position,winner,dnf_flag,driver_avg_finish_last3
0,1046,2020,16,2020-12-06,Sakhir Grand Prix,bahrain,851,aitken,Aitken,3,Williams,17,16,0.0,1,18.0,0,1,NaN
1,1010,2019,1,2019-03-17,Australian Grand Prix,albert_park,848,albon,Albon,5,Toro Rosso,13,14,0.0,11,13.0,0,0,NaN
2,1011,2019,2,2019-03-31,Bahrain Grand Prix,bahrain,848,albon,Albon,5,Toro Rosso,12,9,2.0,1,12.0,0,0,14.00
3,1013,2019,4,2019-04-28,Azerbaijan Grand Prix,baku,848,albon,Albon,5,Toro Rosso,11,11,0.0,11,12.0,0,0,11.50
4,1014,2019,5,2019-05-12,Spanish Grand Prix,catalunya,848,albon,Albon,5,Toro Rosso,11,11,0.0,1,12.0,0,0,11.33


### Driver Momentum

Average points scored over last 3 races.

In [5]:
df['driver_avg_points_last3'] = df.groupby('driver_name')['points'].transform(
                                 lambda x : round(x.shift(1).rolling(3,min_periods=1).mean(),2)
                                 )
df.head()

,race_id,year,round,race_date,race_name,circuit_id,driver_id,driver_code,driver_name,constructor_id,constructor_name,grid_position,finish_position,points,status_id,quali_position,winner,dnf_flag,driver_avg_finish_last3,driver_avg_points_last3
0,1046,2020,16,2020-12-06,Sakhir Grand Prix,bahrain,851,aitken,Aitken,3,Williams,17,16,0.0,1,18.0,0,1,NaN,NaN
1,1010,2019,1,2019-03-17,Australian Grand Prix,albert_park,848,albon,Albon,5,Toro Rosso,13,14,0.0,11,13.0,0,0,NaN,NaN
2,1011,2019,2,2019-03-31,Bahrain Grand Prix,bahrain,848,albon,Albon,5,Toro Rosso,12,9,2.0,1,12.0,0,0,14.00,0.00
3,1013,2019,4,2019-04-28,Azerbaijan Grand Prix,baku,848,albon,Albon,5,Toro Rosso,11,11,0.0,11,12.0,0,0,11.50,1.00
4,1014,2019,5,2019-05-12,Spanish Grand Prix,catalunya,848,albon,Albon,5,Toro Rosso,11,11,0.0,1,12.0,0,0,11.33,0.67


### Constructor Form

Average constructor points over last 3 races.

In [6]:
constructor_points = (df.groupby(['race_id', 'constructor_name'])['points'].sum().reset_index())
constructor_points = constructor_points.sort_values(by = ["constructor_name", "race_id"])
constructor_points["constructor_avg_points_last3"] = constructor_points.groupby("constructor_name")["points"].transform(
                                                     lambda x : round(x.shift(1).rolling(3, min_periods=1).mean(),2)
                                                     )
constructor_points.head()

,race_id,constructor_name,points,constructor_avg_points_last3
210,1010,Alfa Romeo,4.0,NaN
220,1011,Alfa Romeo,6.0,4.0
230,1012,Alfa Romeo,2.0,5.0
240,1013,Alfa Romeo,1.0,4.0
250,1014,Alfa Romeo,0.0,3.0


In [7]:
df = df.merge(constructor_points[[ "race_id", "constructor_name", "constructor_avg_points_last3" ]], 
              on = ['race_id', 'constructor_name'], how = 'left')
df.head()

,race_id,year,round,race_date,race_name,circuit_id,driver_id,driver_code,driver_name,constructor_id,...,grid_position,finish_position,points,status_id,quali_position,winner,dnf_flag,driver_avg_finish_last3,driver_avg_points_last3,constructor_avg_points_last3
0,1046,2020,16,2020-12-06,Sakhir Grand Prix,bahrain,851,aitken,Aitken,3,...,17,16,0.0,1,18.0,0,1,NaN,NaN,0.00
1,1010,2019,1,2019-03-17,Australian Grand Prix,albert_park,848,albon,Albon,5,...,13,14,0.0,11,13.0,0,0,NaN,NaN,0.33
2,1011,2019,2,2019-03-31,Bahrain Grand Prix,bahrain,848,albon,Albon,5,...,12,9,2.0,1,12.0,0,0,14.00,0.00,0.33
3,1013,2019,4,2019-04-28,Azerbaijan Grand Prix,baku,848,albon,Albon,5,...,11,11,0.0,11,12.0,0,0,11.50,1.00,1.00
4,1014,2019,5,2019-05-12,Spanish Grand Prix,catalunya,848,albon,Albon,5,...,11,11,0.0,1,12.0,0,0,11.33,0.67,0.67


In [8]:
df.columns

Index(['race_id', 'year', 'round', 'race_date', 'race_name', 'circuit_id',
       'driver_id', 'driver_code', 'driver_name', 'constructor_id',
       'constructor_name', 'grid_position', 'finish_position', 'points',
       'status_id', 'quali_position', 'winner', 'dnf_flag',
       'driver_avg_finish_last3', 'driver_avg_points_last3',
       'constructor_avg_points_last3'],
      dtype='object')

### Driver Win Rate at Circuit

In [9]:
df["driver_circuit_win"] = np.where(df['winner'] == 1, 1, 0)

df["driver_win_rate_at_circuit"] = df.groupby(["driver_name", "circuit_id"])['driver_circuit_win'].transform(
                                   lambda x : x.shift(1).expanding().mean()
                                   )

### Rolling DNF Rate

In [10]:
df["driver_dnf_rate"] = ( df.groupby("driver_name")["dnf_flag"].transform( 
                        lambda x: x.shift(1).rolling(5, min_periods=1).mean() ) 
                        )
df.head()

,race_id,year,round,race_date,race_name,circuit_id,driver_id,driver_code,driver_name,constructor_id,...,status_id,quali_position,winner,dnf_flag,driver_avg_finish_last3,driver_avg_points_last3,constructor_avg_points_last3,driver_circuit_win,driver_win_rate_at_circuit,driver_dnf_rate
0,1046,2020,16,2020-12-06,Sakhir Grand Prix,bahrain,851,aitken,Aitken,3,...,1,18.0,0,1,NaN,NaN,0.00,0,NaN,NaN
1,1010,2019,1,2019-03-17,Australian Grand Prix,albert_park,848,albon,Albon,5,...,11,13.0,0,0,NaN,NaN,0.33,0,NaN,NaN
2,1011,2019,2,2019-03-31,Bahrain Grand Prix,bahrain,848,albon,Albon,5,...,1,12.0,0,0,14.00,0.00,0.33,0,NaN,0.0
3,1013,2019,4,2019-04-28,Azerbaijan Grand Prix,baku,848,albon,Albon,5,...,11,12.0,0,0,11.50,1.00,1.00,0,NaN,0.0
4,1014,2019,5,2019-05-12,Spanish Grand Prix,catalunya,848,albon,Albon,5,...,1,12.0,0,0,11.33,0.67,0.67,0,NaN,0.0


### Championship Momentum

In [11]:
df['driver_points_so_far'] = df.groupby(['year','driver_name'])['points'].transform(
                              lambda x : x.shift(1).cumsum()
                              )

In [12]:
constructor_race_points = (df.groupby(['year','race_id','constructor_name'])['points'].sum().reset_index())
constructor_race_points['constructor_points_so_far'] = constructor_race_points.groupby(['year','constructor_name'])['points'].transform(
                                                       lambda x : x.shift(1).cumsum()
                                                       )
df = df.merge(constructor_race_points[ [ "year", "race_id", "constructor_name", "constructor_points_so_far" ] ], 
               on=["year", "race_id", "constructor_name"], how="left")
df.head()

,race_id,year,round,race_date,race_name,circuit_id,driver_id,driver_code,driver_name,constructor_id,...,winner,dnf_flag,driver_avg_finish_last3,driver_avg_points_last3,constructor_avg_points_last3,driver_circuit_win,driver_win_rate_at_circuit,driver_dnf_rate,driver_points_so_far,constructor_points_so_far
0,1046,2020,16,2020-12-06,Sakhir Grand Prix,bahrain,851,aitken,Aitken,3,...,0,1,NaN,NaN,0.00,0,NaN,NaN,NaN,0.0
1,1010,2019,1,2019-03-17,Australian Grand Prix,albert_park,848,albon,Albon,5,...,0,0,NaN,NaN,0.33,0,NaN,NaN,NaN,NaN
2,1011,2019,2,2019-03-31,Bahrain Grand Prix,bahrain,848,albon,Albon,5,...,0,0,14.00,0.00,0.33,0,NaN,0.0,0.0,1.0
3,1013,2019,4,2019-04-28,Azerbaijan Grand Prix,baku,848,albon,Albon,5,...,0,0,11.50,1.00,1.00,0,NaN,0.0,2.0,3.0
4,1014,2019,5,2019-05-12,Spanish Grand Prix,catalunya,848,albon,Albon,5,...,0,0,11.33,0.67,0.67,0,NaN,0.0,2.0,3.0


### Keep Core Predictive Features

In [13]:
df.columns

Index(['race_id', 'year', 'round', 'race_date', 'race_name', 'circuit_id',
       'driver_id', 'driver_code', 'driver_name', 'constructor_id',
       'constructor_name', 'grid_position', 'finish_position', 'points',
       'status_id', 'quali_position', 'winner', 'dnf_flag',
       'driver_avg_finish_last3', 'driver_avg_points_last3',
       'constructor_avg_points_last3', 'driver_circuit_win',
       'driver_win_rate_at_circuit', 'driver_dnf_rate', 'driver_points_so_far',
       'constructor_points_so_far'],
      dtype='object')

In [14]:
feature_cols = [
    'grid_position',
    'quali_position',
    'driver_avg_finish_last3', 
    'driver_avg_points_last3',
    'constructor_avg_points_last3', 
    'driver_win_rate_at_circuit', 
    'driver_dnf_rate',
    'driver_points_so_far', 
    'constructor_points_so_far'
]

df[feature_cols].head()

,grid_position,quali_position,driver_avg_finish_last3,driver_avg_points_last3,constructor_avg_points_last3,driver_win_rate_at_circuit,driver_dnf_rate,driver_points_so_far,constructor_points_so_far
0,17,18.0,NaN,NaN,0.00,NaN,NaN,NaN,0.0
1,13,13.0,NaN,NaN,0.33,NaN,NaN,NaN,NaN
2,12,12.0,14.00,0.00,0.33,NaN,0.0,0.0,1.0
3,11,12.0,11.50,1.00,1.00,NaN,0.0,2.0,3.0
4,11,12.0,11.33,0.67,0.67,NaN,0.0,2.0,3.0


In [15]:
print(df.shape)
df[feature_cols].isnull().sum()

(2976, 26)


grid_position                     0
quali_position                    0
driver_avg_finish_last3          40
driver_avg_points_last3          40
constructor_avg_points_last3     32
driver_win_rate_at_circuit      947
driver_dnf_rate                  40
driver_points_so_far            152
constructor_points_so_far       140
dtype: int64

In [16]:
# Filing Missing values of divers_avg_finish_last3 as 20 
# 20 ≈ worst finish
# safer neutral assumption
df['driver_avg_finish_last3'] = (df["driver_avg_finish_last3"].fillna(20))

In [17]:
df[feature_cols].isnull().sum()

grid_position                     0
quali_position                    0
driver_avg_finish_last3           0
driver_avg_points_last3          40
constructor_avg_points_last3     32
driver_win_rate_at_circuit      947
driver_dnf_rate                  40
driver_points_so_far            152
constructor_points_so_far       140
dtype: int64

In [18]:
# replacing other columns NAN value with "0" will be best solution
remaining_cols = ['driver_avg_points_last3',
                  'constructor_avg_points_last3',
                  'driver_win_rate_at_circuit',
                  'driver_dnf_rate',
                  'driver_points_so_far',
                  'constructor_points_so_far' ]
df[remaining_cols] = df[remaining_cols].fillna(0)
df[feature_cols].isnull().sum()

grid_position                   0
quali_position                  0
driver_avg_finish_last3         0
driver_avg_points_last3         0
constructor_avg_points_last3    0
driver_win_rate_at_circuit      0
driver_dnf_rate                 0
driver_points_so_far            0
constructor_points_so_far       0
dtype: int64

### Encode Categorical Variables

In [19]:
ml_df = df[feature_cols]
ml_df.dtypes

grid_position                     int64
quali_position                  float64
driver_avg_finish_last3         float64
driver_avg_points_last3         float64
constructor_avg_points_last3    float64
driver_win_rate_at_circuit      float64
driver_dnf_rate                 float64
driver_points_so_far            float64
constructor_points_so_far       float64
dtype: object

We also need three more columns : 
- driver_name
- constructor_name
- circuit_id

Here I am using `LabelEncoder`
- I know LabelEncoder is used for target labels.
- but tree-based models like XGBoost handle categorical encodings differently from linear models.

In [20]:
le_drivers = LabelEncoder()
le_constructor = LabelEncoder()
le_circuit = LabelEncoder()

In [21]:
df['driver_encoded'] = le_drivers.fit_transform(df['driver_name'])
df['constructor_encoded'] = le_constructor.fit_transform(df['constructor_name'])
df['circuit_encoded'] = le_circuit.fit_transform(df['circuit_id'])

In [22]:
le_drivers.classes_

array(['Aitken', 'Albon', 'Alonso', 'Bearman', 'Bottas', 'Colapinto',
       'Doohan', 'Ericsson', 'Fittipaldi', 'Gasly', 'Giovinazzi',
       'Grosjean', 'Hamilton', 'Hartley', 'Hülkenberg', 'Kubica', 'Kvyat',
       'Latifi', 'Lawson', 'Leclerc', 'Magnussen', 'Mazepin', 'Norris',
       'Ocon', 'Piastri', 'Pérez', 'Ricciardo', 'Russell', 'Räikkönen',
       'Sainz', 'Sargeant', 'Schumacher', 'Sirotkin', 'Stroll', 'Tsunoda',
       'Vandoorne', 'Verstappen', 'Vettel', 'Zhou', 'de Vries'],
      dtype=object)

In [23]:
feature_cols.extend([
    "driver_encoded",
    "constructor_encoded",
    "circuit_encoded"
])
ml_df = df[feature_cols + ['winner','race_id']].copy()

In [24]:
print("ML Dataset Shape:", ml_df.shape)
ml_df.head()

ML Dataset Shape: (2976, 14)


,grid_position,quali_position,driver_avg_finish_last3,driver_avg_points_last3,constructor_avg_points_last3,driver_win_rate_at_circuit,driver_dnf_rate,driver_points_so_far,constructor_points_so_far,driver_encoded,constructor_encoded,circuit_encoded,winner,race_id
0,17,18.0,20.00,0.00,0.00,0.0,0.0,0.0,0.0,0,15,2,0,1046
1,13,13.0,20.00,0.00,0.33,0.0,0.0,0.0,0.0,1,14,0,0,1010
2,12,12.0,14.00,0.00,0.33,0.0,0.0,0.0,1.0,1,14,2,0,1011
3,11,12.0,11.50,1.00,1.00,0.0,0.0,2.0,3.0,1,14,3,0,1013
4,11,12.0,11.33,0.67,0.67,0.0,0.0,2.0,3.0,1,14,4,0,1014


### Feature Correlation with Winner

In [25]:
corr = ml_df.corr(numeric_only=True)['winner'].sort_values(ascending=False)
print(corr)

winner                          1.000000
driver_avg_points_last3         0.442675
constructor_avg_points_last3    0.387964
driver_points_so_far            0.363131
constructor_points_so_far       0.304960
driver_win_rate_at_circuit      0.246233
driver_encoded                  0.079822
constructor_encoded             0.077094
circuit_encoded                -0.000006
race_id                        -0.000104
driver_dnf_rate                -0.163415
grid_position                  -0.301838
driver_avg_finish_last3        -0.327750
quali_position                 -0.328386
Name: winner, dtype: float64


In [26]:
ml_df.to_csv("../data/processed/f1_ml_dataset.csv", index=False)
print("Feature engineered dataset saved.")

Feature engineered dataset saved.


### Saving the encoder
so that we can use the same encoders in predict.ipynb

In [28]:
import joblib
joblib.dump( le_drivers, "../models/le_driver.pkl" ) 
joblib.dump( le_constructor, "../models/le_constructor.pkl" ) 
joblib.dump( le_circuit, "../models/le_circuit.pkl" )

['../models/le_circuit.pkl']